In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
#读取行业信息数据
stock_industry = pd.read_excel(r"E:\A股leverage研究\raw data\data_info.xlsx",skiprows=[0,2]) #去掉英文代码以及单位信息

#将证券代码填充到六位
stock_industry['证券代码'] = stock_industry['证券代码'].apply(lambda x: str(x).zfill(6)) #经检查无重复的证券代码

#选择需要的列
stock_industry = stock_industry[['证券代码','证券简称','上市日期','行业代码D','公司活动情况']]



In [3]:
#读取股票收益率数据
stock_returns = pd.read_excel(r"E:\A股leverage研究\raw data\data_return.xlsx",skiprows=[0,2]) #去掉英文代码以及单位信息

#将证券代码填充到六位
stock_returns['证券代码'] = stock_returns['证券代码'].apply(lambda x: str(x).zfill(6)).astype('str') 

#将交易月份转换为datetime格式
stock_returns['交易月份'] = pd.to_datetime(stock_returns['交易月份'], format='%Y-%m')

# #将交易月份对齐到月初
# stock_returns['交易月份'] = stock_returns['交易月份']- pd.offsets.MonthBegin(1)

#选择需要的列
stock_returns = stock_returns[['证券代码','交易月份','月收盘价','月个股流通市值','月个股总市值','考虑现金红利再投资的月个股回报率','市场类型']]

#重命名部分列
stock_returns.rename(columns={'考虑现金红利再投资的月个股回报率':'月个股回报率'}, inplace=True)

#将月个股流通市值与月个股总市值的单位从千元换算成元
stock_returns['月个股流通市值'] = stock_returns['月个股流通市值'] * 1000
stock_returns['月个股总市值'] = stock_returns['月个股总市值'] * 1000

#对市场类型筛选，只取所有的A股（1：主板A股，4：深证A股，64：北证A股）
stock_returns  = stock_returns.astype({'市场类型': 'str'}).query("市场类型 in ['1','4','16','32','64']").reset_index(drop=True)


#添加decdate列，用于与财务数据合并
stock_returns = stock_returns.assign(
    DecDate=lambda x: pd.to_datetime({
        'year': x['交易月份'].dt.year - 1 - (x['交易月份'].dt.month < 7).astype(int),
        'month': 12,
        'day': 1
    })
)

In [4]:
stock_returns

,证券代码,交易月份,月收盘价,月个股流通市值,月个股总市值,月个股回报率,市场类型,DecDate
0,000001,1991-04-01,43.68,1.157520e+09,2.118487e+09,NaN,4,1989-12-01
1,000001,1991-05-01,38.34,1.016010e+09,1.859497e+09,-0.122253,4,1989-12-01
2,000001,1991-06-01,33.99,9.007350e+08,1.648521e+09,-0.113459,4,1989-12-01
3,000001,1991-07-01,29.54,7.828100e+08,1.432695e+09,-0.130921,4,1990-12-01
4,000001,1991-08-01,15.00,6.748338e+08,1.346275e+09,-0.411588,4,1990-12-01
...,...,...,...,...,...,...,...,...
855074,920992,2025-06-01,21.26,5.931152e+08,2.056500e+09,0.039101,64,2023-12-01
855075,920992,2025-07-01,22.44,6.260351e+08,2.170642e+09,0.055503,64,2024-12-01
855076,920992,2025-08-01,21.65,6.039955e+08,2.094225e+09,-0.035205,64,2024-12-01
855077,920992,2025-09-01,21.07,5.878146e+08,2.038121e+09,-0.026790,64,2024-12-01


In [5]:
stock_returns.duplicated().sum()  #检查是否有重复行，结果为0

np.int64(0)

In [6]:
#取出月个股市值
stock_me = stock_returns[['证券代码','交易月份','月个股总市值','月个股流通市值']].copy()

#六月市值
stock_me_jun = (
    stock_me.query('交易月份.dt.month==6')
    .assign(交易月份 = lambda x: pd.to_datetime((x['交易月份'].dt.year-1).astype(str) + '-12-01'))
    .loc[:, ['证券代码','交易月份','月个股总市值']]
    .rename(columns={'交易月份': 'DecDate','月个股总市值': 'mejun'})
)

#十二月市值
stock_me_dec = (
    stock_me.query('交易月份.dt.month==12')
    .assign(交易月份 = lambda x: pd.to_datetime(x['交易月份'].dt.year.astype(str) + '-12-01'))
    .loc[:, ['证券代码','交易月份','月个股总市值']]
    .rename(columns={'交易月份': 'DecDate','月个股总市值': 'medec'})
)

In [7]:
stock_me_jun

,证券代码,DecDate,mejun
2,000001,1990-12-01,1.648521e+09
14,000001,1991-12-01,5.657258e+09
26,000001,1992-12-01,7.220400e+09
38,000001,1993-12-01,3.620977e+09
50,000001,1994-12-01,3.952899e+09
...,...,...,...
855025,920985,2023-12-01,1.618561e+09
855037,920985,2024-12-01,3.181415e+09
855050,920992,2022-12-01,8.657419e+08
855062,920992,2023-12-01,8.318860e+08


In [8]:
stock_me_dec

,证券代码,DecDate,medec
8,000001,1991-12-01,2.634211e+09
20,000001,1992-12-01,5.994000e+09
32,000001,1993-12-01,6.021490e+09
44,000001,1994-12-01,4.517599e+09
56,000001,1995-12-01,3.326126e+09
...,...,...,...
855019,920985,2023-12-01,2.751243e+09
855031,920985,2024-12-01,3.094762e+09
855044,920992,2022-12-01,1.127883e+09
855056,920992,2023-12-01,1.238156e+09


In [9]:
#读取财务数据
stock_finance = pd.read_excel(r"E:\A股leverage研究\raw data\data_financial.xlsx",skiprows=[0,2]) #去掉英文代码以及单位信息

#将证券代码填充到六位
stock_finance['证券代码'] = stock_finance['证券代码'].apply(lambda x: str(x).zfill(6)) 

#将交易月份转换为datetime格式
stock_finance['统计截止日期'] = pd.to_datetime(stock_finance['统计截止日期'])

#筛选报表类型为合并报表（type为A）的数据
stock_finance = stock_finance[stock_finance['报表类型'] == 'A'].reset_index(drop=True)

#将优先股的NaN值填充为0
stock_finance['其中：优先股'] = stock_finance['其中：优先股'].fillna(0)

#删除统计截至日期为1月初的数据
stock_finance = stock_finance[stock_finance.统计截止日期.dt.month != 1].reset_index(drop=True)

#计算账面价值（book equity = 所有者权益合计-优先股）
stock_finance ['账面价值'] = stock_finance['归属于母公司所有者权益合计'] - stock_finance['其中：优先股']
stock_finance.loc[stock_finance['账面价值'] < 0, '账面价值'] = np.nan


In [10]:
stock_finance

,证券代码,证券简称,统计截止日期,报表类型,资产总计,负债合计,其中：优先股,归属于母公司所有者权益合计,所有者权益合计,账面价值
0,000001,深发展A,1990-12-31,A,2.919190e+09,5.628000e+07,0.0,2.365100e+08,2.365100e+08,2.365100e+08
1,000001,深发展A,1991-12-31,A,4.354460e+09,7.723000e+07,0.0,5.779600e+08,5.779600e+08,5.779600e+08
2,000001,深发展A,1992-12-31,A,7.522847e+09,1.958932e+08,0.0,5.456622e+08,5.456622e+08,5.456622e+08
3,000001,深发展A,1993-12-31,A,9.337871e+09,8.148741e+09,0.0,1.189130e+09,1.189130e+09,1.189130e+09
4,000001,深发展A,1994-06-30,A,1.246595e+10,1.095304e+10,0.0,1.512913e+09,1.512913e+09,1.512913e+09
...,...,...,...,...,...,...,...,...,...,...
295885,920992,中科美菱,2024-09-30,A,7.287716e+08,1.237962e+08,0.0,6.049754e+08,6.049754e+08,6.049754e+08
295886,920992,中科美菱,2024-12-31,A,7.452764e+08,1.335353e+08,0.0,6.117411e+08,6.117411e+08,6.117411e+08
295887,920992,中科美菱,2025-03-31,A,7.438311e+08,1.264821e+08,0.0,6.173490e+08,6.173490e+08,6.173490e+08
295888,920992,中科美菱,2025-06-30,A,7.291701e+08,1.130226e+08,0.0,6.161475e+08,6.161475e+08,6.161475e+08


In [11]:
#将统计截止日期调整为月初日期
stock_finance['统计截止日期'] = stock_finance['统计截止日期'] - pd.offsets.MonthBegin(1)

In [12]:
stock_finance

,证券代码,证券简称,统计截止日期,报表类型,资产总计,负债合计,其中：优先股,归属于母公司所有者权益合计,所有者权益合计,账面价值
0,000001,深发展A,1990-12-01,A,2.919190e+09,5.628000e+07,0.0,2.365100e+08,2.365100e+08,2.365100e+08
1,000001,深发展A,1991-12-01,A,4.354460e+09,7.723000e+07,0.0,5.779600e+08,5.779600e+08,5.779600e+08
2,000001,深发展A,1992-12-01,A,7.522847e+09,1.958932e+08,0.0,5.456622e+08,5.456622e+08,5.456622e+08
3,000001,深发展A,1993-12-01,A,9.337871e+09,8.148741e+09,0.0,1.189130e+09,1.189130e+09,1.189130e+09
4,000001,深发展A,1994-06-01,A,1.246595e+10,1.095304e+10,0.0,1.512913e+09,1.512913e+09,1.512913e+09
...,...,...,...,...,...,...,...,...,...,...
295885,920992,中科美菱,2024-09-01,A,7.287716e+08,1.237962e+08,0.0,6.049754e+08,6.049754e+08,6.049754e+08
295886,920992,中科美菱,2024-12-01,A,7.452764e+08,1.335353e+08,0.0,6.117411e+08,6.117411e+08,6.117411e+08
295887,920992,中科美菱,2025-03-01,A,7.438311e+08,1.264821e+08,0.0,6.173490e+08,6.173490e+08,6.173490e+08
295888,920992,中科美菱,2025-06-01,A,7.291701e+08,1.130226e+08,0.0,6.161475e+08,6.161475e+08,6.161475e+08


In [13]:
#取出资产总计(total assets)和账面价值(book equity)两列
stock_at_be = stock_finance[['证券代码','统计截止日期','资产总计','账面价值']]

######be和at在每个财年保持不变，这里参考美股的做法，取年度的at和be数据######
#增加年份列
stock_at_be['年份'] = stock_at_be['统计截止日期'].dt.year

#取年报数据
stock_at_be = (stock_at_be
    .sort_values(by=['证券代码','统计截止日期'])
    .groupby(['证券代码','年份'])
    .apply(lambda x: x[x['统计截止日期'].dt.month == 12])
    .reset_index(drop=True)
)

#将统计截止日期改写成对应年度的十二月初的数据，用于与收益率数据合并
stock_at_be = stock_at_be.rename(columns={'统计截止日期':'DecDate'})

#删除年份列
stock_at_be = stock_at_be.drop(columns=['年份'])



In [14]:
# ######lt需要季度的数据######
# ###填充版
# #取出负债合计(total liabilities)
# stock_lt = stock_finance[['证券代码','统计截止日期','负债合计']]

# #对数据进行扩充
# def fill_quarterly_data(df):
#     """
#     填充季度数据
    
#     参数:
#     df: DataFrame，包含三列：证券代码、统计截止日期、负债合计
    
#     返回:
#     填充后的DataFrame
#     """
#     # 获取列名
#     col_code = df.columns[0]  # 证券代码
#     col_date = df.columns[1]  # 统计截止日期
#     col_debt = df.columns[2]  # 负债合计
    
#     result_list = []
    
#     # 按证券代码分组处理
#     for code, group in df.groupby(col_code):
#         group = group.copy().sort_values(by=col_date)
        
#         # 获取该证券的所有年份
#         years = group[col_date].dt.year.unique()
        
#         for year in sorted(years):
#             # 获取该年度的数据
#             year_data = group[group[col_date].dt.year == year].copy()
#             year_data['month_day'] = year_data[col_date].dt.strftime('%m-%d')
            
#             # 获取该年度已有的日期
#             existing_dates = set(year_data['month_day'].tolist())
            
#             # 定义标准的四个季度
#             q1 = '03-01'
#             q2 = '06-01'
#             q3 = '09-01'
#             q4 = '12-01'
            
#             # 判断是否需要填充
#             # 如果已经有全部四个季度的数据，则不需要填充
#             has_all_quarters = {q1, q2, q3, q4}.issubset(existing_dates)
            
#             if not has_all_quarters:
#                 # 需要填充
#                 # Q1 (3.01) 用 Q2 (6.01) 的数据填充
#                 if q1 not in existing_dates and q2 in existing_dates:
#                     if year > 1994:  # 1994年不需要补充3.01
#                         source_row = year_data[year_data['month_day'] == q2].iloc[0]
#                         new_row = source_row.copy()
#                         new_row[col_date] = pd.to_datetime(f'{year}-{q1}')
#                         new_row['month_day'] = q1
#                         result_list.append(new_row)
                
#                 # Q3 (9.01) 用 Q4 (12.01) 的数据填充
#                 if q3 not in existing_dates and q4 in existing_dates:
#                     if year < 2025:  # 2025年无需补充，但9.01应该已经存在
#                         source_row = year_data[year_data['month_day'] == q4].iloc[0]
#                         new_row = source_row.copy()
#                         new_row[col_date] = pd.to_datetime(f'{year}-{q3}')
#                         new_row['month_day'] = q3
#                         result_list.append(new_row)
#                     # elif year == 1994:  # 1994年特殊处理：补充9.01
#                     #     if q4 in existing_dates:
#                     #         source_row = year_data[year_data['month_day'] == q4].iloc[0]
#                     #         new_row = source_row.copy()
#                     #         new_row[col_date] = pd.to_datetime(f'{year}-{q3}')
#                     #         new_row['month_day'] = q3
#                     #         result_list.append(new_row)
            
#             # 添加原有数据
#             for _, row in year_data.iterrows():
#                 result_list.append(row)
    
#     # 转换为DataFrame
#     result_df = pd.DataFrame(result_list)
    
#     # 删除辅助列
#     if 'month_day' in result_df.columns:
#         result_df = result_df.drop(columns=['month_day'])
    
#     # 去重并排序
#     result_df = result_df.drop_duplicates(subset=[col_code, col_date])
#     result_df = result_df.sort_values(by=[col_code, col_date])
#     result_df = result_df.reset_index(drop=True)
    
#     return result_df

# stock_lt_cleaned = fill_quarterly_data(stock_lt)
# stock_lt_cleaned = stock_lt_cleaned.rename(columns={'统计截止日期':'id'})

In [15]:
# #####lt需要季度的数据######
# ## 非填充版
# #取出负债合计(total liabilities)
# stock_lt = stock_finance[['证券代码','统计截止日期','负债合计']]

# def build_quarterly_lt(df):
#     col_code = df.columns[0]
#     col_date = df.columns[1]
#     col_value = df.columns[2]
#     df_sorted = df.sort_values(by=[col_code, col_date]).reset_index(drop=True)

#     placeholder_rows = []
#     for code, group in df_sorted.groupby(col_code):
#         for year, year_data in group.groupby(group[col_date].dt.year):
#             existing_months = set(year_data[col_date].dt.month)
#             for month in (3, 9): 
#                 if month not in existing_months:
#                     placeholder_rows.append({
#                         col_code: code,
#                         col_date: pd.Timestamp(year=int(year), month=month, day=1),
#                         col_value: np.nan
#                     })

#     if placeholder_rows:
#         placeholder_df = pd.DataFrame(placeholder_rows)
#         result_df = pd.concat([df_sorted, placeholder_df], ignore_index=True)
#     else:
#         result_df = df_sorted.copy()

#     result_df = (result_df
#         .drop_duplicates(subset=[col_code, col_date], keep='first')
#         .sort_values(by=[col_code, col_date])
#         .reset_index(drop=True))
#     result_df = result_df.rename(columns={col_date: 'id'})
#     return result_df

# stock_lt_cleaned = build_quarterly_lt(stock_lt)


In [16]:
# #####lt需要季度的数据######
# ## 前一季度填充版
# #取出负债合计(total liabilities)
stock_lt = stock_finance[['证券代码','统计截止日期','负债合计']]
def build_quarterly_lt_fill_next(df):
    col_code = df.columns[0]
    col_date = df.columns[1]
    col_value = df.columns[2]

    df_sorted = df.sort_values(by=[col_code, col_date]).reset_index(drop=True)

    # 生成每个股票在每年 3/6/9/12 的占位行
    placeholder_rows = []
    for code, group in df_sorted.groupby(col_code):
        for year, year_data in group.groupby(group[col_date].dt.year):
            existing_months = set(year_data[col_date].dt.month)
            for month in (3, 6, 9, 12):
                if month not in existing_months:
                    placeholder_rows.append({
                        col_code: code,
                        col_date: pd.Timestamp(year=int(year), month=month, day=1),
                        col_value: np.nan
                    })

    if placeholder_rows:
        placeholder_df = pd.DataFrame(placeholder_rows)
        result_df = pd.concat([df_sorted, placeholder_df], ignore_index=True)
    else:
        result_df = df_sorted.copy()

    result_df = (result_df
        .drop_duplicates(subset=[col_code, col_date], keep='first')
        .sort_values(by=[col_code, col_date])
        .reset_index(drop=True))

    # 用上一季度值填充缺失（按股票分组）
    result_df[col_value] = result_df.groupby(col_code)[col_value].ffill()

    result_df = result_df.rename(columns={col_date: 'id'})
    return result_df
stock_lt_cleaned = build_quarterly_lt_fill_next(stock_lt)

In [17]:
stock_returns

,证券代码,交易月份,月收盘价,月个股流通市值,月个股总市值,月个股回报率,市场类型,DecDate
0,000001,1991-04-01,43.68,1.157520e+09,2.118487e+09,NaN,4,1989-12-01
1,000001,1991-05-01,38.34,1.016010e+09,1.859497e+09,-0.122253,4,1989-12-01
2,000001,1991-06-01,33.99,9.007350e+08,1.648521e+09,-0.113459,4,1989-12-01
3,000001,1991-07-01,29.54,7.828100e+08,1.432695e+09,-0.130921,4,1990-12-01
4,000001,1991-08-01,15.00,6.748338e+08,1.346275e+09,-0.411588,4,1990-12-01
...,...,...,...,...,...,...,...,...
855074,920992,2025-06-01,21.26,5.931152e+08,2.056500e+09,0.039101,64,2023-12-01
855075,920992,2025-07-01,22.44,6.260351e+08,2.170642e+09,0.055503,64,2024-12-01
855076,920992,2025-08-01,21.65,6.039955e+08,2.094225e+09,-0.035205,64,2024-12-01
855077,920992,2025-09-01,21.07,5.878146e+08,2.038121e+09,-0.026790,64,2024-12-01


In [18]:
#与收益率数据合并
df = pd.merge(stock_returns, stock_industry, how='left', on='证券代码')

# #删除上市后前6个月的数据
# df['月末日期'] = df['交易月份'] + pd.offsets.MonthEnd(0)
# df = df[df['上市日期'] <= (df['月末日期'] - pd.DateOffset(months=6))].reset_index(drop=True)
# df = df.drop(columns=['月末日期'])
df['交易月份'] = pd.to_datetime(df['交易月份'])
df['上市日期'] = pd.to_datetime(df['上市日期'])
trade_m = df['交易月份'].dt.to_period('M')
list_m = df['上市日期'].dt.to_period('M')

months_diff = (trade_m.dt.year - list_m.dt.year) * 12 + (trade_m.dt.month - list_m.dt.month)

df = df[months_diff >= 6].reset_index(drop=True)

#剔除金融行业
df = df[~df['行业代码D'].astype(str).str.startswith('J')].reset_index(drop=True)

#去掉ST股
df = df[~df['证券简称'].astype(str).str.contains('ST')].reset_index(drop=True)

#去掉PT股
df = df[~df['证券简称'].astype(str).str.contains('PT')].reset_index(drop=True)

# #去掉非正常经营公司
# df = df[df['公司活动情况'] == 'A'].reset_index(drop=True)

#合并me_jun
df = pd.merge(df, stock_me_jun, how='left', on=['证券代码','DecDate'])

#合并me_dec
df = pd.merge(df, stock_me_dec, how='left', on=['证券代码','DecDate'])

#合并at_be
df = pd.merge(df, stock_at_be, how='left', on=['证券代码','DecDate'])



In [19]:
df['year'] = df['交易月份'].dt.year
df['month'] = df['交易月份'].dt.month
def gen_id(row):
    year = row['year']
    month = row['month']
    
    # 计算当前季度的月份范围
    if month in [1,2,3]:
        return  str(year-1)+'-12'+'-01'
    elif month in [4,5,6]:
        return str(year)+'-3'+'-01'
    elif month in [7,8,9]:
        return str(year)+'-6'+'-01'
    elif month in [10,11,12]:
        return str(year)+'-9'+'-01'

df = df.astype({'year':int,'month':int}).assign(id=lambda x: x.apply(gen_id, axis=1)).assign(id =lambda x: pd.to_datetime(x['id']))

#合并lt
df = pd.merge(df, stock_lt_cleaned, how='left', on=['证券代码','id'])

#删除辅助列
df = df.drop(columns=['year','month','id'])

In [20]:
#读取无风险利率-数据来源为中央财经大学fama五因子数据
df_rf = pd.read_csv(r"E:\A股leverage研究\raw data\fivefactor_monthly.csv")

df_rf = df_rf[['trdmn','rf']].rename(columns={'trdmn':'交易月份'}).assign(交易月份 = lambda x: pd.to_datetime(x['交易月份'], format='%Y%m') - pd.offsets.MonthBegin(1))

df = pd.merge(df, df_rf, how='left', on='交易月份')

#计算超额收益率
df['ret_excess'] = df['月个股回报率'] - df['rf']

#计算账面市值比bm
df['bm'] = df['账面价值'] / df['medec']

#计算杠杆率leverage
df['lev'] = df['负债合计'] / (df['负债合计']+df['月个股总市值'])




In [21]:
df.lev.value_counts(dropna=False)

lev
NaN         568
0.000000    251
0.446820      3
0.104321      3
0.621270      3
           ... 
0.098883      1
0.123655      1
0.138302      1
0.148928      1
0.146932      1
Name: count, Length: 738025, dtype: int64

In [22]:
df

,证券代码,交易月份,月收盘价,月个股流通市值,月个股总市值,月个股回报率,市场类型,DecDate,证券简称,上市日期,...,公司活动情况,mejun,medec,资产总计,账面价值,负债合计,rf,ret_excess,bm,lev
0,000002,1991-07-01,6.15,347586760.0,4.733686e+08,-0.061069,4,1990-12-01,万科A,1991-01-29,...,A,5.041568e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000002,1991-08-01,6.30,356064490.0,4.849142e+08,0.024390,4,1990-12-01,万科A,1991-01-29,...,A,5.041568e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,000002,1991-09-01,4.70,265635410.0,3.617614e+08,-0.253968,4,1990-12-01,万科A,1991-01-29,...,A,5.041568e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000002,1991-10-01,16.90,955157120.0,1.300802e+09,2.595745,4,1990-12-01,万科A,1991-01-29,...,A,5.041568e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,000002,1991-11-01,17.00,960808940.0,1.308499e+09,0.005917,4,1990-12-01,万科A,1991-01-29,...,A,5.041568e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
742449,920992,2025-06-01,21.26,593115240.0,2.056500e+09,0.039101,64,2023-12-01,中科美菱,2022-10-18,...,A,8.318860e+08,1.238156e+09,7.328239e+08,6.015200e+08,1.264821e+08,0.000788,0.038313,0.485819,0.057940
742450,920992,2025-07-01,22.44,626035090.0,2.170642e+09,0.055503,64,2024-12-01,中科美菱,2022-10-18,...,A,2.056500e+09,1.324246e+09,7.452764e+08,6.117411e+08,1.130226e+08,0.000788,0.054715,0.461954,0.049492
742451,920992,2025-08-01,21.65,603995530.0,2.094225e+09,-0.035205,64,2024-12-01,中科美菱,2022-10-18,...,A,2.056500e+09,1.324246e+09,7.452764e+08,6.117411e+08,1.130226e+08,0.000788,-0.035993,0.461954,0.051205
742452,920992,2025-09-01,21.07,587814590.0,2.038121e+09,-0.026790,64,2024-12-01,中科美菱,2022-10-18,...,A,2.056500e+09,1.324246e+09,7.452764e+08,6.117411e+08,1.130226e+08,0.000788,-0.027578,0.461954,0.052541


In [23]:
#添加滞后一期的杠杆率
lev_lag = (df
    .assign(
        交易月份=lambda x: x["交易月份"] + pd.DateOffset(months=1),
        lev_lag=lambda x: x["lev"]
    )
    .get(["证券代码", "交易月份", "lev_lag"])
)
# 将滞后杠杆率合并回主表
df = df.merge(lev_lag, how="left", on=["证券代码", "交易月份"])

In [24]:
df.lev_lag.value_counts(dropna=False)   

lev_lag
NaN         9614
0.000000     250
0.446820       3
0.287642       3
0.328471       3
            ... 
0.085386       1
0.098883       1
0.123655       1
0.138302       1
0.060974       1
Name: count, Length: 728999, dtype: int64

In [25]:
#添加滞后一期的总市值
me_lag_t = (df
    .assign(
        交易月份=lambda x: x["交易月份"] + pd.DateOffset(months=1),
        me_lag_t=lambda x: x["月个股总市值"]
    )
    .get(["证券代码", "交易月份", "me_lag_t"])
)
# 将滞后总市值合并回主表
df = df.merge(me_lag_t, how="left", on=["证券代码", "交易月份"])

In [26]:
#添加滞后一期的流通市值
me_lag_f = (df
    .assign(
        交易月份=lambda x: x["交易月份"] + pd.DateOffset(months=1),
        me_lag_f=lambda x: x["月个股流通市值"]
    )
    .get(["证券代码", "交易月份", "me_lag_f"])
)
# 将滞后流通市值合并回主表
df = df.merge(me_lag_f, how="left", on=["证券代码", "交易月份"])

In [27]:
# # 如果 date 是 datetime，其中 day 不重要，可以先转成年月
# df['year_month'] = df['交易月份'].dt.to_period('M')

# # 按月份分组，每月计算上月个股总市值的 30% 分位数
# quantile_30 = df.groupby('year_month')['me_lag_t'].quantile(0.3)

# # 将 quantile merge 回原表
# df = df.merge(quantile_30.rename('q30'), left_on='year_month', right_index=True)

# # 保留上月总市值 > 上月 30% 分位数的股票
# df = df[df['me_lag_t'] > df['q30']].copy()

# # 删除辅助列
# df.drop(columns=['q30', 'year_month'], inplace=True)

# # 1) 用上月为分组键（假设 交易月份 是当月）
# df['year_month_lag'] = (df['交易月份'] - pd.offsets.MonthBegin(1)).dt.to_period('M')

# # 2) 在“上月横截面”计算 30% 分位数
# quantile_30 = df.groupby('year_month_lag')['me_lag_t'].quantile(0.3)

# # 3) 把阈值合并回当月样本（按上月键）
# df = df.merge(quantile_30.rename('q30'), left_on='year_month_lag', right_index=True)

# # 4) 用上月阈值筛选当月样本
# df = df[df['me_lag_t'] > df['q30']].copy()

# # 5) 清理辅助列
# df.drop(columns=['q30', 'year_month_lag'], inplace=True)

In [28]:
df = df[df['交易月份'] >= '1994-07-01'].reset_index(drop=True)
# df = df[df['交易月份'] >= '2000-01-01'].reset_index(drop=True)

In [29]:
df

,证券代码,交易月份,月收盘价,月个股流通市值,月个股总市值,月个股回报率,市场类型,DecDate,证券简称,上市日期,...,资产总计,账面价值,负债合计,rf,ret_excess,bm,lev,lev_lag,me_lag_t,me_lag_f
0,000002,1994-07-01,4.75,6.092602e+08,8.884321e+08,-0.096958,4,1993-12-01,万科A,1991-01-29,...,2.136159e+09,9.284174e+08,1.205853e+09,0.008719,-0.105677,0.373321,0.575783,0.550700,9.838216e+08,6.746756e+08
1,000002,1994-08-01,7.05,9.042705e+08,1.318620e+09,0.484211,4,1993-12-01,万科A,1991-01-29,...,2.136159e+09,9.284174e+08,1.205853e+09,0.008719,0.475492,0.373321,0.477665,0.575783,8.884321e+08,6.092602e+08
2,000002,1994-09-01,8.67,1.112060e+09,1.621622e+09,0.229787,4,1993-12-01,万科A,1991-01-29,...,2.136159e+09,9.284174e+08,1.205853e+09,0.008719,0.221068,0.373321,0.426477,0.477665,1.318620e+09,9.042705e+08
3,000002,1994-10-01,6.25,8.016582e+08,1.168990e+09,-0.279123,4,1993-12-01,万科A,1991-01-29,...,2.136159e+09,9.284174e+08,1.205853e+09,0.008719,-0.287842,0.373321,0.507761,0.426477,1.621622e+09,1.112060e+09
4,000002,1994-11-01,6.20,7.952450e+08,1.159638e+09,-0.008000,4,1993-12-01,万科A,1991-01-29,...,2.136159e+09,9.284174e+08,1.205853e+09,0.008719,-0.016719,0.373321,0.509769,0.507761,1.168990e+09,8.016582e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740992,920992,2025-06-01,21.26,5.931152e+08,2.056500e+09,0.039101,64,2023-12-01,中科美菱,2022-10-18,...,7.328239e+08,6.015200e+08,1.264821e+08,0.000788,0.038313,0.485819,0.057940,0.060069,1.979115e+09,5.707967e+08
740993,920992,2025-07-01,22.44,6.260351e+08,2.170642e+09,0.055503,64,2024-12-01,中科美菱,2022-10-18,...,7.452764e+08,6.117411e+08,1.130226e+08,0.000788,0.054715,0.461954,0.049492,0.057940,2.056500e+09,5.931152e+08
740994,920992,2025-08-01,21.65,6.039955e+08,2.094225e+09,-0.035205,64,2024-12-01,中科美菱,2022-10-18,...,7.452764e+08,6.117411e+08,1.130226e+08,0.000788,-0.035993,0.461954,0.051205,0.049492,2.170642e+09,6.260351e+08
740995,920992,2025-09-01,21.07,5.878146e+08,2.038121e+09,-0.026790,64,2024-12-01,中科美菱,2022-10-18,...,7.452764e+08,6.117411e+08,1.130226e+08,0.000788,-0.027578,0.461954,0.052541,0.051205,2.094225e+09,6.039955e+08


In [30]:
# df.to_excel(r"E:\A股leverage研究\processed data\data_used_raw.xlsx", index=False)
# df.to_csv(r"E:\A股leverage研究\processed data\data_used_raw.csv", index=False)
df.to_parquet(r"E:\A股leverage研究\processed data\data_used_raw.parquet", index=False)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 740997 entries, 0 to 740996
Data columns (total 24 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   证券代码        740997 non-null  object        
 1   交易月份        740997 non-null  datetime64[ns]
 2   月收盘价        740997 non-null  float64       
 3   月个股流通市值     740997 non-null  float64       
 4   月个股总市值      740997 non-null  float64       
 5   月个股回报率      740997 non-null  float64       
 6   市场类型        740997 non-null  object        
 7   DecDate     740997 non-null  datetime64[ns]
 8   证券简称        740997 non-null  object        
 9   上市日期        740997 non-null  datetime64[ns]
 10  行业代码D       740997 non-null  object        
 11  公司活动情况      740997 non-null  object        
 12  mejun       721508 non-null  float64       
 13  medec       700315 non-null  float64       
 14  资产总计        718118 non-null  float64       
 15  账面价值        713081 non-null  float64       
 16  负债

In [32]:
df.市场类型.value_counts()

市场类型
1     305961
4     282247
16    117837
32     26336
64      8616
Name: count, dtype: int64